# Episode 29: Deployment

Your agent runs in a notebook. To make it a *product*, it has to be reachable — by people, or by other software.

**In this episode you'll build:** an agent exposed over the **A2A protocol** so other agents and services can call it as a standard, vendor-neutral endpoint.

This closes **Group 6 — Production**.

## Who is the consumer?

Deployment depends on who calls your agent:

| Consumer | Reach for | Covered in |
|---|---|---|
| A human, in a browser | AG-UI streaming UI | Episode 21 |
| A human, minimal frontend | A plain HTTP + HTML endpoint | Episode 22 |
| **Another agent or service** | **A2A protocol** | **This episode** |

Episodes 21 and 22 served *humans*. A2A serves *machines* — it's how one agent calls another across process, host, or organization boundaries.

## What A2A is

**A2A (Agent2Agent)** is an open, spec-defined protocol for agent-to-agent communication. Instead of inventing a bespoke HTTP API, you expose your agent over A2A and any A2A client can:

- discover the agent's capabilities from a published **agent card**
- send messages and receive replies over a standard wire format (JSON-RPC, REST, or gRPC)
- track long-running tasks by id

The agent card lives at `/.well-known/agent-card.json` — a client reads it and knows exactly how to talk to your agent.

## Install

A2A is an optional extra.

In [1]:
!pip install "ag2[a2a]" uvicorn -q

## Build the A2A server

Three objects: `A2AServer` wraps the agent, `build_card` declares its capabilities, and `server.build_jsonrpc(...)` produces an ASGI app you serve.

We *build and inspect* it here. The card carries the agent's name, description, and skills; the app exposes the agent-card route plus the JSON-RPC endpoint.

In [2]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.a2a import A2AServer, build_card
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


agent = Agent(
    "weather_service",
    prompt="Answer weather questions.",
    config=config,
    tools=[get_weather],
)

server = A2AServer(agent)  # wrap the agent
card = build_card(agent, url="http://127.0.0.1:8000")  # declare its capabilities
app = server.build_jsonrpc(url="http://127.0.0.1:8000", card=card)  # ASGI app to serve

print("A2AServer wraps:", agent.name)
print("agent card:")
print("  name:       ", card.name)
print("  description:", card.description)
print("  skills:     ", [s.name for s in card.skills])
print("JSON-RPC app: ", type(app).__name__)
print("  routes:     ", [r.path for r in app.routes])

A2AServer wraps: weather_service
agent card:
  name:        weather_service
  description: Answer weather questions.
  skills:      ['weather_service']
JSON-RPC app:  Starlette
  routes:      ['/', '/.well-known/agent-card.json', '/.well-known/agent.json']


## Run it as a server

The cell above *builds* the app. To serve it, run it with `uvicorn` from a terminal — save this as `a2a_server.py` and run `python a2a_server.py`.

```python
# a2a_server.py
from dotenv import load_dotenv
load_dotenv()

import uvicorn

from autogen.beta import Agent
from autogen.beta.a2a import A2AServer, build_card
from autogen.beta.config import OpenAIConfig


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


agent = Agent("weather_service", prompt="Answer weather questions.",
              config=OpenAIConfig(model="gpt-4.1-mini"), tools=[get_weather])

server = A2AServer(agent)
card = build_card(agent, url="http://127.0.0.1:8000")
app = server.build_jsonrpc(url="http://127.0.0.1:8000", card=card)

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
```

## Connecting a client

Any A2A client can now reach the agent. From another AG2 agent, `A2AConfig` makes a remote A2A endpoint look like a local model config:

```python
from autogen.beta import Agent
from autogen.beta.a2a import A2AConfig

remote = Agent(
    "client",
    config=A2AConfig(card_url="http://127.0.0.1:8000"),
)
reply = await remote.ask("What's the weather in Paris?")
```

The client reads the card, picks a transport, and the call routes to your server — the calling code looks exactly like talking to any other agent.

## Choosing a deployment

| You want… | Use | Episode |
|---|---|---|
| A rich streaming chat UI for users | `AGUIStream` | 21 |
| A minimal custom web frontend | Plain Starlette + HTML | 22 |
| Other agents/services to call yours | `A2AServer` | 29 |

They aren't exclusive — one agent can be served over AG-UI *and* A2A at once, from the same `Agent` object.

## Additional content

**Three transports, one server.** `A2AServer` can also build REST (`build_rest`) and gRPC (`build_grpc`) bindings. They share one task store, so a task created over JSON-RPC is visible over REST. JSON-RPC is the recommended default.

**Long-running tasks.** A2A models work as *tasks* with ids — a client can submit a task, disconnect, and poll or stream results later. Good for agent work that outlives a single HTTP request.

**A2A vs the Hub network.** A2A is for crossing *process and org boundaries* with a vendor-neutral protocol. The in-process Hub network (Group 3) is for tightly-coordinated agents in one runtime. Different problems — use A2A when the agents genuinely don't share a process.

## What's next

Group 6 is complete — your agents are observable, safe, tested, affordable, and deployable. Group 7 goes **advanced**: Episode 30 starts with **redundancy** — running work in parallel and reconciling the results.